# 02 – Baseline CNN (Trained from Scratch)
**Project:** AI Football Event Classifier (CNN) — Yannick Maas

This notebook builds a **simple custom CNN** as a quick baseline before moving to transfer learning.  
It predicts both the **event** label and the **view** label simultaneously (multi-task).

Run `01_data_preparation.ipynb` first so that `outputs/manifest.csv` exists.

In [2]:
import os, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cuda


In [3]:
OUTPUT_DIR    = Path('outputs')
MANIFEST_PATH = OUTPUT_DIR / 'manifest.csv'
IMG_SIZE      = (224, 224)
BATCH_SIZE    = 32
NUM_EPOCHS    = 10
LR            = 1e-3

In [4]:
df = pd.read_csv(MANIFEST_PATH)
NUM_EVENT_CLASSES = df['event_idx'].nunique()
NUM_VIEW_CLASSES  = df['view_idx'].nunique()
print(f'Total images : {len(df):,}  |  Event classes: {NUM_EVENT_CLASSES}  |  View classes: {NUM_VIEW_CLASSES}')

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df   = df[df['split'] == 'val'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)

Total images : 2,100  |  Event classes: 2  |  View classes: 14


In [5]:
train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class FootballDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, int(row['event_idx']), int(row['view_idx'])

train_loader = DataLoader(FootballDataset(train_df, train_transform), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(FootballDataset(val_df,   val_transform),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(FootballDataset(test_df,  val_transform),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print('DataLoaders ready')

DataLoaders ready


In [6]:
class SimpleCNN(nn.Module):
    def __init__(self, num_event_classes, num_view_classes):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3,  32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128,256,3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.AdaptiveAvgPool2d(4),
        )
        self.flatten    = nn.Flatten()
        self.dropout    = nn.Dropout(0.4)
        fc_in = 256 * 4 * 4
        self.event_head = nn.Linear(fc_in, num_event_classes)
        self.view_head  = nn.Linear(fc_in, num_view_classes)

    def forward(self, x):
        x = self.dropout(self.flatten(self.backbone(x)))
        return self.event_head(x), self.view_head(x)

model = SimpleCNN(NUM_EVENT_CLASSES, NUM_VIEW_CLASSES).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

Parameters: 454,928


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'val_event_acc': [], 'val_view_acc': []}


def run_epoch(loader, training=True):
    phase = 'train' if training else 'validation'
    model.train() if training else model.eval()
    total_loss = event_correct = view_correct = total = 0
    progress_freq = max(1, len(loader) // 5)
    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for batch_idx, (imgs, ev, vw) in enumerate(loader, start=1):
            imgs, ev, vw = imgs.to(DEVICE), ev.to(DEVICE), vw.to(DEVICE)
            e_out, v_out = model(imgs)
            loss = criterion(e_out, ev) + criterion(v_out, vw)
            if training:
                optimizer.zero_grad(); loss.backward(); optimizer.step()

            total_loss    += loss.item() * len(imgs)
            event_correct += (e_out.argmax(1) == ev).sum().item()
            view_correct  += (v_out.argmax(1) == vw).sum().item()
            total         += len(imgs)

            if training and batch_idx % progress_freq == 0:
                print(f'    [{phase}] batch {batch_idx}/{len(loader)} | batch_loss={loss.item():.4f}')

    avg_loss = total_loss / total
    event_acc = event_correct / total
    view_acc = view_correct / total
    print(f'  Completed {phase} pass: loss={avg_loss:.4f}, event_acc={event_acc:.3f}, view_acc={view_acc:.3f}')
    return avg_loss, event_acc, view_acc

print(f'Training for {NUM_EPOCHS} epochs …')
print(f'  Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')
for epoch in range(1, NUM_EPOCHS + 1):
    print(f'\nEpoch {epoch}/{NUM_EPOCHS}')
    tl, _, _         = run_epoch(train_loader, training=True)
    vl, ev_acc, vw_acc = run_epoch(val_loader, training=False)
    scheduler.step()
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['val_event_acc'].append(ev_acc)
    history['val_view_acc'].append(vw_acc)
    print(f'  Epoch {epoch} summary -> train_loss={tl:.4f}, val_loss={vl:.4f}, event_acc={ev_acc:.3f}, view_acc={vw_acc:.3f}')

torch.save(model.state_dict(), OUTPUT_DIR / 'baseline_cnn.pth')
print('Model saved → outputs/baseline_cnn.pth')

Training 10 epochs …


In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs, history['train_loss'], label='train')
axes[0].plot(epochs, history['val_loss'],   label='val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(epochs, history['val_event_acc'], label='event acc')
axes[1].plot(epochs, history['val_view_acc'],  label='view acc')
axes[1].set_title('Val Accuracy'); axes[1].legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'baseline_training_curves.png', dpi=120)
plt.show()

In [ ]:
model.eval()
all_et, all_ep = [], []
with torch.no_grad():
    for imgs, ev, vw in test_loader:
        e_out, _ = model(imgs.to(DEVICE))
        all_et.extend(ev.numpy())
        all_ep.extend(e_out.argmax(1).cpu().numpy())

event_classes = sorted(df['event_label'].unique())
print(classification_report(all_et, all_ep, target_names=event_classes))

cm = confusion_matrix(all_et, all_ep)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=event_classes, yticklabels=event_classes)
plt.title('Confusion Matrix – Baseline CNN')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'baseline_confusion_matrix.png', dpi=120)
plt.show()

## Summary
This baseline gives a lower-bound reference.  
Identify which classes are hardest before proceeding to transfer learning.

➡️ Continue with **`03_transfer_learning.ipynb`**